# LlamaCloud Extract V2: Complete Walkthrough

This notebook demonstrates the full capabilities of LlamaCloud's Extract V2 API, from simple receipt extraction to advanced features like citations with bounding box visualization, confidence scores, and agentic extraction.

## Section 1: Setup

In [ ]:
!apt-get update -qq > /dev/null 2>&1 && apt-get install -y -qq poppler-utils > /dev/null 2>&1
%pip install -q llama-cloud pdf2image Pillow matplotlib

In [ ]:
import os
import json
import time
from getpass import getpass

import httpx

from llama_cloud import LlamaCloud

# Load API key from environment or prompt for it
API_KEY = os.environ.get("LLAMA_CLOUD_API_KEY") or getpass("Enter your LlamaCloud API key: ")
BASE_URL = "https://api.cloud.llamaindex.ai"

client = LlamaCloud(api_key=API_KEY, base_url=BASE_URL)
print("Client initialized.")

# Note: All SDK methods accept an optional project_id parameter.
# If omitted, the API uses your key's default project.
# For multi-project setups, pass project_id explicitly:
#   client.files.create(file=f, purpose="extract", project_id="your-project-id")
#   client.extract.run(document_input_value=file_id, project_id="your-project-id", ...)

### Upload test files

We upload three documents: a receipt, a US patent (the Transformer paper, 16 pages), and a SaaS pitch deck slide.

In [ ]:
import urllib.request
from pathlib import Path

# Download test files from GitHub (needed for Colab)
REPO = "run-llama/llama-cloud-py"
BRANCHES = ["main", "eli/extract-v2-cookbook"]
FILE_PREFIX = "examples/extract"

test_files = {
    "receipt": "test-files/receipt/noisebridge_receipt.pdf",
    "patent": "test-files/patent/US10452978_transformer.pdf",
    "slide": "test-files/slide/saas_slide.pdf",
}
schema_files = {
    "receipt": "test-files/receipt/schema.json",
    "patent": "test-files/patent/schema.json",
    "slide": "test-files/slide/schema.json",
}

for path in list(test_files.values()) + list(schema_files.values()):
    local = Path(path)
    if not local.exists():
        local.parent.mkdir(parents=True, exist_ok=True)
        for branch in BRANCHES:
            url = f"https://raw.githubusercontent.com/{REPO}/{branch}/{FILE_PREFIX}/{path}"
            try:
                urllib.request.urlretrieve(url, local)
                print(f"Downloaded {path}")
                break
            except urllib.error.HTTPError:
                continue

# Upload to LlamaCloud
file_ids = {}
for name, path in test_files.items():
    with open(path, "rb") as f:
        uploaded = client.files.create(file=f, purpose="extract")
    file_ids[name] = uploaded.id
    print(f"{name}: {uploaded.name} -> {uploaded.id}")

print("\nAll files uploaded.")

---
## Section 2: Schema Generation & Validation

V2 can generate schemas from a natural language prompt, from a sample document, or both. You can also validate schemas before running extraction.

In [ ]:
# Generate a schema from a natural language prompt
schema_from_prompt = client.extract.generate_schema(
    prompt="Extract the company name, total revenue, and list of line items with description and amount from an invoice",
)

print(f"Generated config name: {schema_from_prompt.name}")
print(f"\nSchema:")
print(json.dumps(schema_from_prompt.parameters.data_schema, indent=2))

In [ ]:
# Generate a schema from a sample document (let the API infer fields)
schema_from_file = client.extract.generate_schema(
    file_id=file_ids["receipt"],
    prompt="Extract all structured data from this receipt",
)

print(f"Generated config name: {schema_from_file.name}")
inferred_schema = schema_from_file.parameters.data_schema
print(f"Inferred fields: {list(inferred_schema.get('properties', {}).keys())}")
print(f"\nSchema:")
print(json.dumps(inferred_schema, indent=2))

In [ ]:
# Validate a schema before using it
validated = client.extract.validate_schema(data_schema=inferred_schema)
print("Schema is valid.")
print(f"Fields: {list(validated.data_schema.get('properties', {}).keys())}")

In [ ]:
# Use the generated schema to run an extraction
inferred_schema_job = client.extract.run(
    document_input_value=file_ids["receipt"],
    configuration={
        "data_schema": validated.data_schema,
        "extraction_target": "per_doc",
        "tier": "cost_effective",
    },
    verbose=True,
)

print("\n--- Extract Result (using generated schema) ---")
print(json.dumps(inferred_schema_job.extract_result, indent=2))

---
## Section 3: Quick Start. Receipt Extraction

Extract structured data from a simple one-page receipt. We show the two-step flow (`create` + `wait_for_completion`) first, then the `run()` shortcut.

In [ ]:
import matplotlib.pyplot as plt
from PIL import ImageDraw
from pdf2image import convert_from_path

receipt_pages = convert_from_path("test-files/receipt/noisebridge_receipt.pdf", dpi=150)
fig, ax = plt.subplots(figsize=(8, 10))
ax.imshow(receipt_pages[0])
ax.set_title("Source Document: Noisebridge Receipt")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
with open("test-files/receipt/schema.json") as f:
    receipt_schema = json.load(f)

print(json.dumps(receipt_schema, indent=2))

In [ ]:
# Two-step flow: create the job, then poll until complete
receipt_job = client.extract.create(
    document_input_value=file_ids["receipt"],
    configuration={
        "data_schema": receipt_schema,
        "extraction_target": "per_doc",
        "tier": "cost_effective",
    },
)
print(f"Created job: {receipt_job.id} (status: {receipt_job.status})")

receipt_job = client.extract.wait_for_completion(receipt_job.id, verbose=True)

print("\n--- Extract Result ---")
print(json.dumps(receipt_job.extract_result, indent=2))

In [ ]:
# Shortcut: run() combines create + wait_for_completion in one call
# We use run() for the rest of this notebook.
receipt_job_quick = client.extract.run(
    document_input_value=file_ids["receipt"],
    configuration={
        "data_schema": receipt_schema,
        "extraction_target": "per_doc",
        "tier": "cost_effective",
    },
    verbose=True,
)
print(f"\nrun() returned: {receipt_job_quick.status}")
print(f"Total: {receipt_job_quick.extract_result.get('total')}")

---
## Section 4: Complex Document. US Patent (16 pages)

Extract 15 fields from the Transformer patent (US10452978), including arrays of inventors, references, and classification codes.

In [ ]:
patent_pages = convert_from_path("test-files/patent/US10452978_transformer.pdf", dpi=150, first_page=1, last_page=1)
fig, ax = plt.subplots(figsize=(10, 12))
ax.imshow(patent_pages[0])
ax.set_title("Source Document: US10452978 (Transformer Patent) - Page 1 of 16")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
with open("test-files/patent/schema.json") as f:
    patent_schema = json.load(f)

print(f"Schema has {len(patent_schema['properties'])} fields")
print(f"\nField names: {list(patent_schema['properties'].keys())}")

In [ ]:
patent_job = client.extract.run(
    document_input_value=file_ids["patent"],
    configuration={"data_schema": patent_schema, "extraction_target": "per_doc", "tier": "cost_effective"},
    verbose=True,
)

print("\n--- Extract Result (all 15 fields) ---")
for field, value in patent_job.extract_result.items():
    display_val = str(value)[:80]
    if isinstance(value, list):
        display_val = f"[{len(value)} items] {display_val}"
    print(f"  {field}: {display_val}")

---
## Section 5: Citations & Bounding Boxes

Enable `cite_sources` to get per-field citations with page numbers, matching text, and bounding box coordinates. Then visualize them on the PDF page images.

In [ ]:
# Run extraction with citations on the patent
citation_schema = {
    "type": "object",
    "properties": {
        "applicant": {"type": "string"},
        "grant_date": {"type": "string"},
        "num_claims": {"type": "number"},
    },
}

citation_job = client.extract.run(
    document_input_value=file_ids["patent"],
    configuration={
        "data_schema": citation_schema,
        "extraction_target": "per_doc",
        "tier": "cost_effective",
        "cite_sources": True,
    },
    verbose=True,
)
print(f"\nExtract result:")
for k, v in citation_job.extract_result.items():
    print(f"  {k}: {v}")

In [ ]:
# Fetch the job with expanded metadata for per-field citations
citation_detail = client.extract.get(citation_job.id, expand=["extract_metadata"])

citation_metadata = citation_detail.extract_metadata.field_metadata.document_metadata
print(f"Fields with metadata: {len(citation_metadata)}")

print("\n--- Per-Field Citations ---\n")
for field_name, field_meta in citation_metadata.items():
    if not isinstance(field_meta, dict):
        continue
    citations = field_meta.get("citation", [])
    if not citations:
        print(f"{field_name}: no citations")
        continue
    for cite in citations:
        print(f"{field_name}:")
        print(f"  page: {cite.get('page')}")
        print(f"  matching_text: {cite.get('matching_text', '')[:80]}")
        print(f"  bounding_boxes: {cite.get('bounding_boxes', [])}")
        print()

### Visualize bounding boxes on PDF pages

Draw red rectangles on the rendered PDF page images to show exactly where each extracted field was found.

In [ ]:
pdf_pages = convert_from_path("test-files/patent/US10452978_transformer.pdf", dpi=150, first_page=1, last_page=1)

def draw_citation_boxes(page_img, field_name, citations):
    """Draw bounding boxes on a PDF page image.
    Coordinates are in absolute PDF points (1/72 inch).
    page_dimensions provides the reference page size for scaling."""
    img = page_img.copy()
    draw = ImageDraw.Draw(img)
    img_w, img_h = img.size

    for cite in citations:
        page_dims = cite.get("page_dimensions", {})
        scale_x = img_w / page_dims.get("width", 612.0)
        scale_y = img_h / page_dims.get("height", 792.0)

        for bbox in cite.get("bounding_boxes", []):
            x = bbox["x"] * scale_x
            y = bbox["y"] * scale_y
            w = bbox["w"] * scale_x
            h = bbox["h"] * scale_y
            draw.rectangle([x, y, x + w, y + h], outline="red", width=3)
            draw.text((x, max(0, y - 15)), field_name, fill="red")

    return img

In [ ]:
# Pick page-1 fields that have bounding boxes (for a varied visualization)
fields_to_show = []
for field_name, field_meta in citation_metadata.items():
    if not isinstance(field_meta, dict):
        continue
    for cite in field_meta.get("citation", []):
        if cite.get("bounding_boxes") and cite.get("page") == 1:
            fields_to_show.append(field_name)
            break
fields_to_show = fields_to_show[:4]
print(f"Visualizing {len(fields_to_show)} fields: {fields_to_show}")

fig, axes = plt.subplots(1, len(fields_to_show), figsize=(6 * len(fields_to_show), 10))
if len(fields_to_show) == 1:
    axes = [axes]

for ax, field_name in zip(axes, fields_to_show):
    field_meta = citation_metadata[field_name]
    citations = field_meta.get("citation", [])
    extracted_val = citation_detail.extract_result.get(field_name)
    page_idx = 0  # all page 1

    annotated = draw_citation_boxes(pdf_pages[page_idx], field_name, citations)
    ax.imshow(annotated)
    ax.set_title(f"{field_name}\n= {str(extracted_val)[:40]}", fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.suptitle("Extract V2: Bounding Box Citations", fontsize=14, y=1.02)
plt.show()

---
## Section 6: Confidence Scores

Enable `confidence_scores` to get a per-field confidence value (0.0 to 1.0) indicating how certain the model is about each extraction.

In [ ]:
confidence_job = client.extract.run(
    document_input_value=file_ids["receipt"],
    configuration={
        "data_schema": receipt_schema,
        "extraction_target": "per_doc",
        "tier": "cost_effective",
        "confidence_scores": True,
    },
    verbose=True,
)

confidence_detail = client.extract.get(confidence_job.id, expand=["extract_metadata"])

confidence_metadata = confidence_detail.extract_metadata.field_metadata.document_metadata
print("\n--- Per-Field Confidence Scores ---\n")
print("Each field has: confidence, parsing_confidence, extraction_confidence\n")
for field_name, field_meta in confidence_metadata.items():
    value = confidence_detail.extract_result.get(field_name)
    if isinstance(field_meta, dict) and "confidence" in field_meta:
        score = field_meta["confidence"]
        print(f"  {field_name}: {score:.4f}  (value: {str(value)[:50]})")
    elif isinstance(field_meta, list):
        print(f"  {field_name}: [array with {len(field_meta)} element(s)]")
        for i, elem in enumerate(field_meta[:2]):
            if isinstance(elem, dict):
                for subfield, sub_meta in elem.items():
                    if isinstance(sub_meta, dict) and "confidence" in sub_meta:
                        print(f"    [{i}].{subfield}: {sub_meta['confidence']:.4f}")

In [ ]:
# Bar chart of confidence scores
field_names = []
scores = []
for field_name, field_meta in confidence_metadata.items():
    if isinstance(field_meta, dict) and "confidence" in field_meta:
        field_names.append(field_name)
        scores.append(float(field_meta["confidence"]))

colors = ["red" if s < 0.8 else "steelblue" for s in scores]

fig, ax = plt.subplots(figsize=(10, max(4, len(field_names) * 0.35)))
ax.barh(field_names, scores, color=colors)
ax.axvline(x=0.8, color="orange", linestyle="--", label="Threshold (0.8)")
ax.set_xlabel("Confidence Score")
ax.set_title("Per-Field Confidence Scores (red = below 0.8)")
ax.set_xlim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
threshold = 0.8
low_confidence = [(n, s) for n, s in zip(field_names, scores) if s < threshold]

if low_confidence:
    print(f"Fields below {threshold} confidence:")
    for name, score in low_confidence:
        print(f"  {name}: {score:.2f}")
else:
    print(f"All fields are at or above {threshold} confidence.")

---
## Section 7: Agentic vs Cost Effective

Compare the two extraction tiers on a SaaS pitch deck slide. The `agentic` tier uses more credits but may produce better results on complex or ambiguous documents.

In [ ]:
with open("test-files/slide/schema.json") as f:
    slide_schema = json.load(f)

slide_pages = convert_from_path("test-files/slide/saas_slide.pdf", dpi=150)
fig, ax = plt.subplots(figsize=(12, 9))
ax.imshow(slide_pages[0])
ax.set_title("Source Document: SaaS Pitch Deck Slide")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
t0 = time.time()
cost_effective_job = client.extract.run(
    document_input_value=file_ids["slide"],
    configuration={"data_schema": slide_schema, "extraction_target": "per_doc", "tier": "cost_effective"},
    verbose=True,
)
cost_effective_time = time.time() - t0

t0 = time.time()
agentic_job = client.extract.run(
    document_input_value=file_ids["slide"],
    configuration={"data_schema": slide_schema, "extraction_target": "per_doc", "tier": "agentic"},
    verbose=True,
)
agentic_time = time.time() - t0

print(f"\nCost effective: {cost_effective_time:.1f}s, Agentic: {agentic_time:.1f}s")

In [ ]:
def flatten_result(result, prefix=""):
    items = {}
    if isinstance(result, dict):
        for k, v in result.items():
            key = f"{prefix}{k}" if not prefix else f"{prefix}.{k}"
            if isinstance(v, dict):
                items.update(flatten_result(v, key))
            else:
                items[key] = v
    return items

cost_effective_flat = flatten_result(cost_effective_job.extract_result)
agentic_flat = flatten_result(agentic_job.extract_result)
all_keys = sorted(set(list(cost_effective_flat.keys()) + list(agentic_flat.keys())))

print(f"{'Field':<40} {'Cost Effective':<30} {'Agentic':<30}")
print("-" * 100)
for key in all_keys:
    ce_val = str(cost_effective_flat.get(key, "--"))[:28]
    ag_val = str(agentic_flat.get(key, "--"))[:28]
    match = "" if ce_val == ag_val else " *"
    print(f"{key:<40} {ce_val:<30} {ag_val:<30}{match}")

print(f"\nTiming: cost_effective={cost_effective_time:.1f}s, agentic={agentic_time:.1f}s")
print("(* = values differ between tiers)")

---
## Section 8: Per-Page Extraction

Extract one result object per page instead of one per document.

In [ ]:
per_page_job = client.extract.run(
    document_input_value=file_ids["patent"],
    configuration={
        "data_schema": patent_schema,
        "extraction_target": "per_page",
        "tier": "cost_effective",
        "target_pages": "1-3",
    },
    verbose=True,
)

results = per_page_job.extract_result
print(f"\nGot {len(results)} page results (type: {type(results).__name__})\n")

for i, page_result in enumerate(results):
    print(f"=== Page {i + 1} ===")
    for field, value in page_result.items():
        if value is not None:
            print(f"  {field}: {str(value)[:60]}")
    print()

---
## Section 9: Advanced Options

Demonstrate `target_pages`, `system_prompt`, and saved configurations.

In [ ]:
# target_pages: only extract from the cover page
target_pages_job = client.extract.run(
    document_input_value=file_ids["patent"],
    configuration={
        "data_schema": patent_schema,
        "extraction_target": "per_doc",
        "tier": "cost_effective",
        "target_pages": "1",
    },
    verbose=True,
)
print("\n--- Page 1 only ---")
for field, value in target_pages_job.extract_result.items():
    if value is not None:
        print(f"  {field}: {str(value)[:60]}")

In [ ]:
# system_prompt: guide extraction behavior
system_prompt_job = client.extract.run(
    document_input_value=file_ids["receipt"],
    configuration={
        "data_schema": receipt_schema,
        "extraction_target": "per_doc",
        "tier": "cost_effective",
        "system_prompt": (
            "All dollar amounts should be formatted as plain numbers without "
            "currency symbols. Dates should be in ISO 8601 format (YYYY-MM-DD)."
        ),
    },
    verbose=True,
)
print("\n--- With system_prompt ---")
print(json.dumps(system_prompt_job.extract_result, indent=2))

### Saved Configurations

Configurations are managed via the `/api/v1/beta/configurations` API. Each configuration has a `product_type` discriminator: `extract_v2` for extract configs, `parse_v2` for parse configs. Once saved, reference them by ID in extract jobs.

In [ ]:
# Configuration API calls (not yet in the SDK)
api_headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

# Create a saved extract configuration
r = httpx.post(
    f"{BASE_URL}/api/v1/beta/configurations",
    headers=api_headers,
    json={
        "name": "cookbook-receipt-extract",
        "parameters": {
            "product_type": "extract_v2",
            "data_schema": receipt_schema,
            "extraction_target": "per_doc",
            "tier": "cost_effective",
        },
    },
    timeout=30,
)
extract_config = r.json()
print(f"Created extract config: {extract_config['id']}")
print(f"  name: {extract_config['name']}, product_type: {extract_config['product_type']}")

# Create a saved parse configuration
r = httpx.post(
    f"{BASE_URL}/api/v1/beta/configurations",
    headers=api_headers,
    json={
        "name": "cookbook-receipt-parse",
        "parameters": {
            "product_type": "parse_v2",
            "version": "latest",
            "tier": "cost_effective",
        },
    },
    timeout=30,
)
parse_config = r.json()
print(f"Created parse config: {parse_config['id']}")
print(f"  name: {parse_config['name']}, product_type: {parse_config['product_type']}")

In [ ]:
# List all saved configurations
r = httpx.get(f"{BASE_URL}/api/v1/beta/configurations", headers=api_headers, timeout=30)
config_list = r.json().get("items", [])

print(f"{'Config ID':<40} {'Name':<30} {'Type'}")
print("-" * 85)
for c in config_list[:5]:
    print(f"{c['id']:<40} {c['name']:<30} {c['product_type']}")

In [ ]:
# Use the saved extract config via configuration_id
saved_config_job = client.extract.run(
    document_input_value=file_ids["receipt"],
    configuration_id=extract_config["id"],
    verbose=True,
)
print("\n--- Extract Result (using saved configuration_id) ---")
print(json.dumps(saved_config_job.extract_result, indent=2))

In [ ]:
# Use the saved parse config via parse_config_id
saved_parse_config_job = client.extract.run(
    document_input_value=file_ids["receipt"],
    configuration={
        "data_schema": receipt_schema,
        "extraction_target": "per_doc",
        "tier": "cost_effective",
        "parse_config_id": parse_config["id"],
    },
    verbose=True,
)
print("\n--- Extract Result (using saved parse_config_id) ---")
print(json.dumps(saved_parse_config_job.extract_result, indent=2))

In [ ]:
# Clean up saved configs
httpx.delete(f"{BASE_URL}/api/v1/beta/configurations/{extract_config['id']}", headers=api_headers, timeout=30)
httpx.delete(f"{BASE_URL}/api/v1/beta/configurations/{parse_config['id']}", headers=api_headers, timeout=30)
print(f"Deleted extract config: {extract_config['id']}")
print(f"Deleted parse config: {parse_config['id']}")

---
## Section 10: Typed Results with ExtractedData

Parse raw extract results into typed Pydantic models using `ExtractedData.from_extract_job`. This gives you type-safe access, automatic confidence parsing, and workflow status tracking.

In [ ]:
from __future__ import annotations

from pydantic import BaseModel

from llama_cloud.types.beta.extracted_data import ExtractedData


# Define a typed schema for the receipt
class ReceiptItem(BaseModel):
    description: str = ""
    quantity: int = 0
    unitPrice: float = 0
    amount: float = 0

class Receipt(BaseModel):
    receiptNumber: str = ""
    invoiceNumber: str = ""
    datePaid: str = ""
    total: float = 0
    subtotal: float = 0
    items: list[ReceiptItem] = []

In [ ]:
# Run extraction with confidence scores so we get field metadata
receipt_confidence_job = client.extract.run(
    document_input_value=file_ids["receipt"],
    configuration={
        "data_schema": receipt_schema,
        "extraction_target": "per_doc",
        "tier": "cost_effective",
        "confidence_scores": True,
    },
    verbose=True,
)

# Get with expanded metadata
receipt_confidence_detail = client.extract.get(receipt_confidence_job.id, expand=["extract_metadata"])

# Parse into typed ExtractedData
extracted = ExtractedData.from_extract_job(receipt_confidence_detail, Receipt)

print(f"Type: {type(extracted.data).__name__}")
print(f"Status: {extracted.status}")
print(f"File ID: {extracted.file_id}")
print(f"\nTyped access:")
print(f"  receipt_number: {extracted.data.receiptNumber}")
print(f"  date_paid: {extracted.data.datePaid}")
print(f"  total: {extracted.data.total}")
print(f"  items: {len(extracted.data.items)} item(s)")
for item in extracted.data.items:
    print(f"    - {item.description}: ${item.amount}")
print(f"\nOverall confidence: {extracted.overall_confidence}")
print(f"Field metadata keys: {list(extracted.field_metadata.keys())}")

---
## Section 11: Job Management

List, inspect, and delete extraction jobs.

In [ ]:
# List recent jobs
jobs_page = client.extract.list(page_size=5)

print(f"{'Job ID':<30} {'Status':<12} {'Created At'}")
print("-" * 70)
for i, job in enumerate(jobs_page):
    if i >= 5:
        break
    print(f"{job.id:<30} {job.status:<12} {job.created_at}")

In [ ]:
# Get a job with expanded configuration
detailed_job = client.extract.get(receipt_job.id, expand=["configuration"])

print(f"Job ID: {detailed_job.id}")
print(f"Status: {detailed_job.status}")
config = detailed_job.configuration
print(f"Configuration:")
print(f"  tier: {config.tier}")
print(f"  extraction_target: {config.extraction_target}")
print(f"  cite_sources: {config.cite_sources}")
print(f"  confidence_scores: {config.confidence_scores}")
print(f"  schema fields: {list(config.data_schema.get('properties', {}).keys())[:5]}...")

In [ ]:
# Delete a job
job_to_delete = target_pages_job.id
print(f"Deleting job: {job_to_delete}")
client.extract.delete(job_to_delete)
print("Deleted successfully.")

---

## Summary

This walkthrough covered:

1. **Setup**: SDK installation, client initialization, file uploads
2. **Schema Generation & Validation**: Generate schemas from prompts or documents, validate before use
3. **Quick Start**: Two-step flow (`create` + `wait_for_completion`) and the `run()` shortcut
4. **Complex Documents**: 15-field patent extraction with arrays and nested types
5. **Citations & Bounding Boxes**: Per-field source citations with visual overlay on PDF pages
6. **Confidence Scores**: Per-field confidence values with threshold filtering
7. **Tier Comparison**: Cost effective vs agentic extraction side-by-side
8. **Per-Page Extraction**: One result per page for multi-page documents
9. **Advanced Options**: target_pages, system_prompt, saved extract and parse configurations
10. **Typed Results**: `ExtractedData.from_extract_job` for Pydantic model parsing with confidence metadata
11. **Job Management**: List, inspect, and delete jobs

For more information, visit the [LlamaCloud documentation](https://docs.cloud.llamaindex.ai).